# 02 · DenPAR Periapical WSP Benchmark (Step-by-Step Validation & Visuals)

Self-contained, resume-safe benchmark notebook using `DenPAR Radiographs Dataset.zip` (official **650 Training / 150 Validation / 200 sealed Testing** splits).

**Run cleanly from top to bottom on a Colab GPU (T4 / L4):**
- **Step 1:** Mount Google Drive, locate `dental-wsp` & `DenPAR Radiographs Dataset.zip`, verify GPU (`Device: cuda`), and set permanent Drive checkpoint folder.
- **Step 2 (`denpar_cls` · Stage 1):** Frozen WSP (`K = 0`, all 6 locked replies) on **Upper (Maxillary) vs. Lower (Mandibular) Arch** across the 150 validation periapicals.
- **Step 3 (`denpar_cls` · Stages 2 & 3):** 3-Arm learning curve (`Frozen WSP` vs. `Random-Init Adam` vs. `WSP Warm-Start Adam`, `K = 2..650`, 5 seeds).
- **Step 4 (`denpar_seg` · Stage 1):** Frozen `periapical-wsp82-v1` (82 params) full mechanistic audit (A/B/C ablations, paired swaps, outside FP share, boundary F1, edge recall, excluded tooth pixels).
- **Step 5 (`denpar_seg` · Stages 2 & 3):** Converged 4-Arm 82-param learning curve (`Frozen WSP`, `Trained Position-Only`, `Random-Init Adam`, `WSP Warm-Start Adam`, `K = 1..200`, 3 seeds, `batch_size=32`, `epochs=150`, cosine LR, Drive-checkpointed after every step).
- **Step 6 (Visuals · Frozen Replies):** Deterministic periapical X-ray overlay grid comparing all 6 frozen replies (`A · Intact` vs. `B · Image-Only`).
- **Step 7 (Visuals · 4 Arms):** Side-by-side periapical X-ray overlay comparing all 4 arms (using median-Dice frozen & warm-start slots at `K = 200`).

**Governance:** Official `Testing` split (`200` films) stays sealed (`split='val'`). Clear outputs before committing to Git.

## Step 1 · Setup, Drive Mount & GPU Check

In [ ]:
from pathlib import Path
import sys

if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_SRC = Path("/content/drive/MyDrive/WSP_Dental_Data/DenPAR Radiographs Dataset.zip")
    candidates = [Path("/content/dental-wsp"), Path("/content/drive/MyDrive/dental-wsp")]
    REPO_DIR = next((p for p in candidates if (p / "benchmarks").is_dir()), candidates[1])
    OUT_ROOT = Path("/content/drive/MyDrive/dental-wsp-runs/denpar")
else:
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "benchmarks").is_dir() else Path.cwd()
    DATA_SRC = REPO_DIR / "data/DenPAR Radiographs Dataset.zip"
    OUT_ROOT = REPO_DIR / "runs/denpar"

assert (REPO_DIR / "benchmarks").is_dir(), f"dental-wsp not found at {REPO_DIR}"
assert DATA_SRC.exists(), f"DenPAR ZIP not found at {DATA_SRC}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

OUT_ROOT.mkdir(parents=True, exist_ok=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from benchmarks.denpar.adapter import verify_denpar_directory
from wsp_core.guards import exposure_guard
exposure_guard(DATA_SRC.parent)
print(f"Repo:        {REPO_DIR}")
print(f"Data:        {DATA_SRC}")
print(f"Checkpoints: {OUT_ROOT}")
print(f"Device:      {DEVICE}")
print(verify_denpar_directory(DATA_SRC))

## Step 2 · `denpar_cls` Stage 1 — Frozen WSP (`K = 0`, Upper vs. Lower Arch, 6 Replies)

In [ ]:
import pandas as pd
from benchmarks.denpar.adapter import load_denpar_cls
from wsp_core.evaluation.classification import evaluate_classifier
from wsp_core.schemas import strict_json_loads, verify_response_lock

train_cls = load_denpar_cls(DATA_SRC, split="train")
val_cls = load_denpar_cls(DATA_SRC, split="val")
print(f"denpar_cls cohorts: train={len(train_cls)} (Upper={sum(s.label for s in train_cls)}), val={len(val_cls)} (Upper={sum(s.label for s in val_cls)})")

cls_design = REPO_DIR / "benchmarks/denpar/locks/denpar_cls.design.lock.json"
cls_resp = REPO_DIR / "benchmarks/denpar/locks/denpar_cls.responses.lock.json"
cls_d_bytes = cls_design.read_bytes()
frozen_cls = verify_response_lock(strict_json_loads(cls_d_bytes), cls_d_bytes, strict_json_loads(cls_resp.read_bytes()), "cls-lenet2d-v1")

cls_stage1_df = pd.DataFrame([{"Reply": slot, **evaluate_classifier(p, val_cls)} for slot, p in frozen_cls.items()]).set_index("Reply")
cls_stage1_df.to_csv(OUT_ROOT / "denpar_cls_stage1_frozen.csv")
display(cls_stage1_df[["roc_auc", "pr_auc", "accuracy", "sensitivity", "specificity", "margin_ablated_roc_auc", "sign_flip_flagged"]].round(4))

## Step 3 · `denpar_cls` Stages 2 & 3 — 3-Arm Learning Curve (`K = 2..650`, 5 Seeds)

In [ ]:
import matplotlib.pyplot as plt
from wsp_core.training.runner import plot_classification_learning_curve, run_classification_three_arm_validation

CLS_OUT = OUT_ROOT / "denpar_cls"
cls_runs, cls_summary = run_classification_three_arm_validation(
    task="denpar_cls",
    train_samples=train_cls,
    val_samples=val_cls,
    design_lock_path=cls_design,
    responses_lock_path=cls_resp,
    budgets=[2, 4, 8, 16, 32, 64, 128, len(train_cls)],
    seeds=[0, 1, 2, 3, 4],
    epochs=200,
    lr=5e-3,
    device=DEVICE,
    out_dir=CLS_OUT,
    resume=True,
)
display(cls_summary.round(4))
fig = plot_classification_learning_curve(cls_summary, task="denpar_cls")
plt.show()
plt.close(fig)
print(f"Saved denpar_cls runs, summary CSV, and PNG to {CLS_OUT}")

## Step 4 · `denpar_seg` Stage 1 — Frozen WSP Mechanistic Audit (`K = 0`, 6 Replies, 150 Validation Films)

In [ ]:
import pandas as pd
from benchmarks.denpar.adapter import load_denpar_seg
from wsp_core.evaluation.segmentation import evaluate_segmentation_paired_audit, evaluate_segmentation_swaps
from wsp_core.schemas import strict_json_loads, verify_response_lock

val_seg = load_denpar_seg(DATA_SRC, split="val")
seg_design = REPO_DIR / "benchmarks/denpar/locks/denpar_seg.design.lock.json"
seg_resp = REPO_DIR / "benchmarks/denpar/locks/denpar_seg.responses.lock.json"
seg_d_bytes = seg_design.read_bytes()
frozen_seg = verify_response_lock(strict_json_loads(seg_d_bytes), seg_d_bytes, strict_json_loads(seg_resp.read_bytes()), "periapical-wsp82-v1")

seg_rows = []
for slot, payload in frozen_seg.items():
    sw = evaluate_segmentation_swaps(payload, val_seg)
    au = evaluate_segmentation_paired_audit(payload, val_seg)
    seg_rows.append({
        "Reply": slot,
        "Intact Dice (%)": sw["full_cohort_dice_a_pct"],
        "Image-only Dice (%)": sw["full_cohort_dice_b_pct"],
        "Position-only Dice (%)": sw["full_cohort_dice_c_pct"],
        "Swap loss A (pp)": sw["intact"]["swap_loss_pp"],
        "Swap groups": f"{sw['groups']}/{sw['total_groups']}",
        "Outside-ROI share of grid-removed FPs (A vs B) (%)": au["outside_fp_reduction_share_pct"],
        "Delta BF1 (pp)": au["bf1_gain_pp"],
        "Delta edge recall (pp)": au["edge_recall_gain_pp"],
        "Box IoU A (%)": au["box_iou_a_pct"],
        "Predicted box valid fraction A (%)": au["box_valid_vol_a_pct"],
        "Excluded tooth pixels A (%)": au["excluded_tooth_a_pct"],
    })
seg_stage1_df = pd.DataFrame(seg_rows).set_index("Reply")
seg_stage1_df.to_csv(OUT_ROOT / "denpar_seg_stage1_frozen.csv")
display(seg_stage1_df.round(4))

## Step 5 · `denpar_seg` Stages 2 & 3 — Converged 4-Arm 82-Param Learning Curve (`K = 1..200`, 3 Seeds)

In [ ]:
import matplotlib.pyplot as plt
from wsp_core.training.runner import plot_segmentation_learning_curve, run_segmentation_three_arm_validation

train_seg = load_denpar_seg(DATA_SRC, split="train")
SEG_OUT = OUT_ROOT / "denpar_seg_converged"
seg_runs, seg_summary = run_segmentation_three_arm_validation(
    task="denpar_seg",
    architecture="periapical-wsp82-v1",
    train_samples=train_seg,
    val_samples=val_seg,
    design_lock_path=seg_design,
    responses_lock_path=seg_resp,
    budgets=[1, 2, 4, 8, 16, 32, 64, 200],
    seeds=[0, 1, 2],
    epochs=150,
    lr=3e-2,
    batch_size=32,
    cosine_lr=True,
    device=DEVICE,
    out_dir=SEG_OUT,
    resume=True,
)
display(seg_summary.round(4))
fig = plot_segmentation_learning_curve(seg_summary, task="denpar_seg")
plt.show()
plt.close(fig)
print(f"Saved converged denpar_seg CSVs, representative payloads, and PNG to {SEG_OUT}")

## Step 6 · Visual X-Ray Comparison 1 — All 6 Frozen Replies (`A · Intact` vs. `B · Image-Only`)

In [ ]:
import matplotlib.pyplot as plt
from wsp_core.visualization import plot_frozen_replies_grid, select_deterministic_examples

ATTR = "Dataset: DenPAR (Rasnayaka et al., Scientific Data 2025, Zenodo) · Deterministic validation sample"
VIZ_DIR = OUT_ROOT / "denpar_seg_visuals"
examples = select_deterministic_examples(val_seg, n=2)

for idx, ex in enumerate(examples, start=1):
    out_p = VIZ_DIR / f"example_{idx}_frozen_6replies.png"
    plot_frozen_replies_grid(
        ex,
        frozen_seg,
        title=f"DenPAR Validation Example {idx} · All 6 Frozen periapical-wsp82-v1 Replies (A Intact vs. B Image-Only)",
        attribution=ATTR,
        out_path=out_p,
    )
    img = plt.imread(out_p)
    plt.figure(figsize=(10, 12)); plt.imshow(img); plt.axis("off"); plt.show()
print(f"Saved 6-reply periapical overlays to {VIZ_DIR}")

## Step 7 · Visual X-Ray Comparison 2 — All 4 Arms Side-by-Side (`K = 200`)

In [ ]:
import json
import matplotlib.pyplot as plt
from wsp_core.visualization import plot_four_arm_comparison

rep_path = OUT_ROOT / "denpar_seg_converged/denpar_seg_representative_payloads.json"
assert rep_path.is_file(), f"Run Step 5 first to generate {rep_path}"
rep_payloads = json.loads(rep_path.read_text(encoding="utf-8"))

for idx, ex in enumerate(examples, start=1):
    out_p = VIZ_DIR / f"example_{idx}_four_arms.png"
    plot_four_arm_comparison(
        ex,
        rep_payloads,
        title=f"DenPAR Validation Example {idx} · 4-Arm Comparison (82-Parameter periapical-wsp82-v1)",
        attribution=ATTR,
        out_path=out_p,
    )
    img = plt.imread(out_p)
    plt.figure(figsize=(11, 8)); plt.imshow(img); plt.axis("off"); plt.show()
print(f"Saved 4-arm periapical comparison overlays to {VIZ_DIR}")

## Step 8 · Review Controls (v0.1.1) — Trivial & Non-LLM Baselines (Validation Only)

Self-contained after Step 1 (no need to re-run Steps 2–7). Evaluates all-foreground, per-image Otsu, train-selected global intensity threshold `t*`, and the human-written 82-parameter center-tap prior (frozen `K=0` and warm-start) for `denpar_seg`, plus constant-score, zero-parameter vertical bright→dark crown-edge / orientation rules, and global-intensity logistic regression for `denpar_cls`, next to frozen WSP mean ± SD.

In [ ]:
from benchmarks.denpar.adapter import load_denpar_cls, load_denpar_seg
from wsp_core.evaluation.baselines import evaluate_classification_baselines, evaluate_segmentation_baselines

if "train_seg" not in globals():
    train_seg = load_denpar_seg(DATA_SRC, split="train")
if "val_seg" not in globals():
    val_seg = load_denpar_seg(DATA_SRC, split="val")
if "train_cls" not in globals():
    train_cls = load_denpar_cls(DATA_SRC, split="train")
if "val_cls" not in globals():
    val_cls = load_denpar_cls(DATA_SRC, split="val")

seg_design = REPO_DIR / "benchmarks/denpar/locks/denpar_seg.design.lock.json"
seg_resp = REPO_DIR / "benchmarks/denpar/locks/denpar_seg.responses.lock.json"
cls_design = REPO_DIR / "benchmarks/denpar/locks/denpar_cls.design.lock.json"
cls_resp = REPO_DIR / "benchmarks/denpar/locks/denpar_cls.responses.lock.json"

CTRL_OUT = OUT_ROOT / "validation_controls"
dp_seg_base_df, dp_seg_base_meta = evaluate_segmentation_baselines(
    task="denpar_seg",
    architecture="periapical-wsp82-v1",
    train_samples=train_seg,
    val_samples=val_seg,
    design_lock_path=seg_design,
    responses_lock_path=seg_resp,
    run_human_warmstart=True,
    budgets=[1, 2, 4, 8, 16, 32, 64, 200],
    seeds=[0, 1, 2],
    epochs=150,
    lr=3e-2,
    batch_size=32,
    device=DEVICE,
    out_dir=CTRL_OUT,
)
print(f"[denpar_seg] Train-selected global threshold t* = {dp_seg_base_meta['global_threshold_t_star']:.4f} (train Dice = {dp_seg_base_meta['global_threshold_train_dice_pct']:.2f}%)")
display(dp_seg_base_df.round(4))

dp_cls_base_df = evaluate_classification_baselines(
    task="denpar_cls",
    train_samples=train_cls,
    val_samples=val_cls,
    budgets=[2, 4, 8, 16, 32, 64, 128, len(train_cls)],
    seeds=[0, 1, 2, 3, 4],
    design_lock_path=cls_design,
    responses_lock_path=cls_resp,
    out_dir=CTRL_OUT,
)
display(dp_cls_base_df.round(4))
print("Non-private aggregate outputs ready to copy to results/validation_controls/:")
for p in sorted(CTRL_OUT.glob("denpar_*")):
    if not p.name.endswith(".private.json"):
        print(" -", p)